# V11 — White-Box Affordance Ladder (Colab T4)

Runs the same demo kernel as the Kaggle path. Use this when Kaggle quota is
exhausted or you want to poke at intermediate state interactively.

**Runtime → Change runtime type → T4 GPU** before running anything. T4 is
Turing (sm_75): fp16 works, bf16 does not, and the kernel forces fp16 on
pre-Ampere hardware automatically.

Modes:
- `forensics` — weight-space audit of an adapter (no GPU needed)
- `adapter-detect` — can a probe tell adapter-active from base, from activations alone?
- `topic-confound` — the H2 rehearsal: does the probe track the installed behaviour, or the domain?
- `organism` — the real run once the organisms ship

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
cap = torch.cuda.get_device_capability()
print('device:', torch.cuda.get_device_name(0), 'sm_%d%d' % cap)
assert cap[0] >= 7, 'Pre-Turing GPU; request a T4'

In [ ]:
# Colab ships its own torch -- do not upgrade it, only what we add on top.
!pip install -q transformers peft accelerate bitsandbytes safetensors scikit-learn

In [ ]:
# Fetch the demo kernel. Two options -- pick one.

# (a) public/authenticated clone of the research repo:
# !git clone https://github.com/SolshineCode/deception-nanochat-sae-research.git
# %cd deception-nanochat-sae-research/experiments/v11_secret_loyalties

# (b) upload kaggle_demo.py directly (works with a private repo):
from google.colab import files
import os
if not os.path.exists('kaggle_demo.py'):
    files.upload()  # select kaggle_demo.py
assert os.path.exists('kaggle_demo.py'), 'kaggle_demo.py not present'

In [ ]:
import importlib, json, sys
import kaggle_demo as K
importlib.reload(K)

K.OUT = '/content/v11_out'
import os; os.makedirs(K.OUT, exist_ok=True)

cfg = dict(K.CONFIG)
cfg['base'] = 'Qwen/Qwen2.5-1.5B-Instruct'
cfg['adapter'] = 'bharati2324/Qwen2.5-1.5B-Instruct-Code-LoRA-r16'
cfg['layers'] = [8, 14, 20, 26]
cfg['n_prompts'] = 40
cfg['max_new_tokens'] = 48
cfg['permutations'] = 500
cfg

## Mode 1 — weight-space forensics (no GPU)

Where does the adapter put its mass, and how many directions does it really use?
Compare `top5_share` against the measured broad-capability baseline (7.3%–12.0%
across two same-task adapters). A narrow behavioural edit concentrating well
above that range is a level-1 detection signal costing zero inference.

In [ ]:
rep = K.mode_forensics(cfg)
for name, r in rep.items():
    print(name)
    print('  modules=%d top5=%.1f%% PR=%.1f top_layers=%s'
          % (r['n_modules'], 100*r['top5_share'], r['participation_ratio'], r['top_layers'][:5]))
    print('  effective ranks (top modules):',
          [round(s['effective_rank'], 2) for s in r['spectra'][:6]])

## Mode 2 — can a probe detect that *something* was installed?

Ground truth is exact: we control whether the adapter is applied. This is a real
affordance-level-1 detection task — the auditor is told nothing about what was
installed or why.

In [ ]:
res = K.mode_adapter_detect(cfg)
print('identical completions base vs adapter: %.1f%%'
      % (100*res['identical_completion_rate']))
for li, cell in res['layers'].items():
    print('L%-3s AUROC=%.3f  p=%.4f  n=%d'
          % (li, cell['auroc']['auroc'], cell['permutation']['p_value'], cell['auroc']['n']))

## Mode 3 — the H2 rehearsal: installed behaviour, or just the domain?

Mapping onto the organism evaluation sets:

| set | here | organism analogue |
|---|---|---|
| D+ | adapter active, coding prompt | activation condition present, correct principal |
| D⁻c | base model, coding prompt | correct principal, activation condition absent |
| D⁻A | adapter active, non-coding prompt | activation present, **wrong principal** |

D⁻A is the discriminator. The modification is installed but should be inert. A
probe that fires there anyway is reading the domain, not the installed
behaviour — the same failure H2 is designed to catch on the organisms.

Every set is scored by a model that did not train on it, and the threshold is
calibrated on held-out negatives. Both matter: skipping either produced a
spurious ~50% firing rate on a D⁻A drawn from exactly the D⁻c distribution.

In [ ]:
res2 = K.mode_topic_confound(cfg)
for li, cell in res2['layers'].items():
    h2 = cell['H2']
    print('L%-3s AUROC=%.3f p=%.4f  %s' % (li, cell['auroc']['auroc'],
                                           cell['permutation']['p_value'], h2.get('verdict')))
    print('      rates', {k: round(v, 3) for k, v in h2.get('positive_rates', {}).items()},
          '| n', h2.get('n_scored'))

## Mode 4 — the real organisms

Point `adapter` at the organism and `trajectories` at the shipped evaluation
sets. Capture writes trajectories plus completions; label those with a judge,
then run `probes_loyalty.py --level 2` for the affordance-gated result.

Watch the `n` reported for D⁻A: the 5% operating point is unstable below ~100
trajectories per set, and the organisms' wrong-principal set is exactly n=100.

In [ ]:
# cfg['adapter'] = '<organism-repo>'
# cfg['trajectories'] = '/content/evalsets.jsonl'
# cfg['quantize_4bit'] = False   # True for the 7B organism on a T4
# res3 = K.mode_organism(cfg)
# res3

In [ ]:
# Persist everything -- 'regeneratable from script' does not count as saved.
import shutil, json, os
summary = {'forensics': rep, 'adapter_detect': res, 'topic_confound': res2}
with open(os.path.join(K.OUT, 'v11_colab_results.json'), 'w') as fh:
    json.dump(summary, fh, indent=2, default=str)
shutil.make_archive('/content/v11_results', 'zip', K.OUT)
from google.colab import files as _f
_f.download('/content/v11_results.zip')